In [1]:
import os
import cv2
import glob
import random
import numpy as np
from tqdm import tqdm
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Conv2DTranspose, 
                                     concatenate, BatchNormalization, Activation, 
                                     add, GlobalAveragePooling2D, Reshape, Dense, 
                                     Multiply, MultiHeadAttention, LayerNormalization)
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

# Cố định seed cho khâu chia tách tập dữ liệu ban đầu
SEED_VALUE = 24
random.seed(SEED_VALUE)
np.random.seed(SEED_VALUE)
tf.random.set_seed(SEED_VALUE)

# GIẢI PHÓNG ĐỘ ĐỘC LẬP PHẦN CỨNG ĐỂ GPU CHẠY SONG SONG ASYNC TỐI ĐA (TĂNG TỐC ĐỘ TRAIN)
if 'TF_DETERMINISTIC_OPS' in os.environ:
    del os.environ['TF_DETERMINISTIC_OPS']

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("⚡ Đã mở khóa tối ưu luồng phần cứng! Tốc độ huấn luyện ảnh màng tế bào sẽ đạt mức tối đa.")

# Siêu tham số tối ưu cho ảnh Grayscale cấu trúc khối ISBI-2012
IMG_HEIGHT = 256       
IMG_WIDTH = 256        
N_CHANNELS = 1         # Ép mạng nhận ảnh xám 1 kênh (Grayscale)
BATCH_SIZE = 4         # Giảm batch size xuống 4 để phù hợp với tập dữ liệu nhỏ 30 ảnh
EPOCHS = 150           
ALPHA = 1.67
LEARNING_RATE = 1e-3

2026-06-22 13:39:00.758635: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782135540.940869      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782135541.006167      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782135541.424789      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782135541.424825      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782135541.424828      23 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
Num GPUs Available: 1
⚡ Đã mở khóa tối ưu luồng phần cứng! Tốc độ huấn luyện ảnh màng tế bào sẽ đạt mức tối đa.


In [2]:
base_path = "/kaggle/input/datasets/quanhh42/multiresunet-datasets/ISBI-2012-challenge"
train_vol_path = os.path.join(base_path, "train-volume.tif")
train_lab_path = os.path.join(base_path, "train-labels.tif")

def load_isbi_data(path, is_mask=False):
    success, images = cv2.imreadmulti(path)
    if not success:
        print(f"⚠️ Không tìm thấy file tại {path}, tạo dữ liệu giả lập...")
        if is_mask: return np.random.randint(0, 2, (30, IMG_HEIGHT, IMG_WIDTH, 1)).astype(np.float32)
        else: return np.random.rand(30, IMG_HEIGHT, IMG_WIDTH, 1).astype(np.float32)
        
    data = []
    for img in images:
        resized = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
        if is_mask:
            resized = resized / 255.0
            resized = np.round(resized, 0)
        else:
            resized = resized / 255.0
        data.append(resized)
    return np.expand_dims(np.array(data, dtype=np.float32), axis=-1)

print("\nĐang tiến hành nạp tập dữ liệu thực tế ISBI-2012...")
X = load_isbi_data(train_vol_path, is_mask=False)
Y = load_isbi_data(train_lab_path, is_mask=True)
print(f"Nạp thành công! X shape: {X.shape}, Y shape: {Y.shape}")

# Phân chia dữ liệu Train-Test bằng train_test_split (80/20) để đảm bảo tính công bằng xuyên suốt 9 nhánh
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=SEED_VALUE)
print(f"✔️ Phân chia hoàn tất: Train = {X_train.shape[0]} ảnh | Val = {X_val.shape[0]} ảnh")


Đang tiến hành nạp tập dữ liệu thực tế ISBI-2012...
Nạp thành công! X shape: (30, 256, 256, 1), Y shape: (30, 256, 256, 1)
✔️ Phân chia hoàn tất: Train = 24 ảnh | Val = 6 ảnh


In [ ]:
def jacard(y_true, y_pred):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    return K.sum(y_true_f * y_pred_f) / (K.sum(y_true_f + y_pred_f - y_true_f * y_pred_f) + K.epsilon())

def dice_coef(y_true, y_pred):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    return (2.0 * K.sum(y_true_f * y_pred_f) + K.epsilon()) / (K.sum(y_true_f) + K.sum(y_pred_f) + K.epsilon())

def ea_ftl_loss(y_true, y_pred):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    tp = K.sum(y_true_f * y_pred_f)
    fp = K.sum((1.0 - y_true_f) * y_pred_f)
    fn = K.sum(y_true_f * (1.0 - y_pred_f))
    tversky = (tp + K.epsilon()) / (tp + 0.3 * fp + 0.7 * fn + K.epsilon())
    ftl = K.pow((1.0 - tversky), 4./3.)
    # Toán tử Sobel xử lý biên hoàn hảo trên ma trận ảnh xám 1 kênh đầu vào
    y_true_edges = tf.image.sobel_edges(y_true)
    y_pred_edges = tf.image.sobel_edges(y_pred)
    edge_loss = K.mean(K.abs(y_true_edges - y_pred_edges))
    return ftl + 0.1 * edge_loss



def conv2d_bn(x, filters, num_row, num_col, padding='same', strides=(1, 1), activation='relu'):
    x = Conv2D(filters, (num_row, num_col), strides=strides, padding=padding, use_bias=False)(x)
    x = BatchNormalization(axis=3, scale=False)(x)
    return Activation(activation)(x) if activation is not None else x

In [4]:
def MultiResBlock_factory(U, inp, use_se=False): 
    W = ALPHA * U
    short = conv2d_bn(inp, int(W*0.167)+int(W*0.333)+int(W*0.5), 1, 1, activation=None)
    c3 = conv2d_bn(inp, int(W*0.167), 3, 3); c5 = conv2d_bn(c3, int(W*0.333), 3, 3); c7 = conv2d_bn(c5, int(W*0.5), 3, 3)
    out = Activation('relu')(add([short, concatenate([c3, c5, c7], 3)]))
    if use_se:
        c_se = K.int_shape(out)[3]
        se = GlobalAveragePooling2D()(out)
        se = Reshape((1, 1, c_se))(se)
        se = Dense(c_se // 8, activation='relu', use_bias=False)(se)
        se = Dense(c_se, activation='sigmoid', use_bias=False)(se)
        out = Multiply()([out, se])
    return out

def ResPath_factory(f, length, inp, use_se=False, use_att_respath=False): 
    out = conv2d_bn(inp, f, 3, 3)
    out = add([conv2d_bn(inp, f, 1, 1, activation=None), out])
    out = Activation('relu')(BatchNormalization(axis=3)(out))
    for _ in range(length - 1):
        short = conv2d_bn(out, f, 1, 1, activation=None)
        out = Activation('relu')(add([short, conv2d_bn(out, f, 3, 3)]))
        out = BatchNormalization(axis=3)(out)
        
    # TÍCH HỢP ĐƯỜNG TRUYỀN SKIP NÂNG CẤP ATTENTION-AWARE RESPATH THEO ĐÚNG SƠ ĐỒ HOÀN THIỆN
    if use_att_respath:
        c_att = K.int_shape(out)[3]
        gating = Conv2D(c_att, (1, 1), padding='same', activation='sigmoid')(out)
        out = Multiply()([out, gating])
        
    if use_se and not use_att_respath: 
        c_se = K.int_shape(out)[3]
        se = GlobalAveragePooling2D()(out)
        se = Reshape((1, 1, c_se))(se)
        se = Dense(c_se // 8, activation='relu', use_bias=False)(se)
        se = Dense(c_se, activation='sigmoid', use_bias=False)(se)
        out = Multiply()([out, se])
    return out

def TransformerBlock_factory(inputs):
    shape = K.int_shape(inputs); h, w, c = shape[1], shape[2], shape[3]
    x = Reshape((h * w, c))(inputs)
    attn_out = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)
    x = LayerNormalization(epsilon=1e-6)(add([x, attn_out]))
    ffn_out = Dense(c)(Dense(512, activation='relu')(x))
    return Reshape((h, w, c))(LayerNormalization(epsilon=1e-6)(add([x, ffn_out])))

os.makedirs('checkpoint_ablation_isbi', exist_ok=True)
print("✔️ Khởi tạo toàn bộ tài nguyên nền tảng và nạp dữ liệu ISBI-2012 thành công!")

✔️ Khởi tạo toàn bộ tài nguyên nền tảng và nạp dữ liệu ISBI-2012 thành công!


In [5]:
# Cấu trúc hàm sinh mô hình linh hoạt động cho ma trận bóc tách thành phần (Ablation Matrix)
def build_ablation_network(use_se, use_transformer, use_att_respath):
    # Khởi tạo Input nhận ma trận ảnh xám 1 kênh màu (192, 256, 1) hoặc (256, 256, 1)
    inputs = Input((IMG_HEIGHT, IMG_WIDTH, N_CHANNELS))
    
    # Encoder
    m1 = MultiResBlock_factory(32, inputs, use_se); p1 = MaxPooling2D((2,2))(m1); r1 = ResPath_factory(32, 4, m1, use_se, use_att_respath)
    m2 = MultiResBlock_factory(64, p1, use_se); p2 = MaxPooling2D((2,2))(m2); r2 = ResPath_factory(64, 3, m2, use_se, use_att_respath)
    m3 = MultiResBlock_factory(128, p2); p3 = MaxPooling2D((2,2))(m3); r3 = ResPath_factory(128, 2, m3, use_se, use_att_respath)
    m4 = MultiResBlock_factory(256, p3); p4 = MaxPooling2D((2,2))(m4); r4 = ResPath_factory(256, 1, m4, use_se, use_att_respath)
    
    # Bottleneck
    m5 = MultiResBlock_factory(512, p4)
    if use_transformer:
        m5 = TransformerBlock_factory(m5)
        
    # Decoder
    u6 = concatenate([Conv2DTranspose(256, (2,2), strides=(2,2), padding='same')(m5), r4], 3); m6 = MultiResBlock_factory(256, u6)
    u7 = concatenate([Conv2DTranspose(128, (2,2), strides=(2,2), padding='same')(m6), r3], 3); m7 = MultiResBlock_factory(128, u7)
    u8 = concatenate([Conv2DTranspose(64, (2,2), strides=(2,2), padding='same')(m7), r2], 3); m8 = MultiResBlock_factory(64, u8, use_se)
    u9 = concatenate([Conv2DTranspose(32, (2,2), strides=(2,2), padding='same')(m8), r1], 3); m9 = MultiResBlock_factory(32, u9, use_se)
    
    return Model(inputs, Conv2D(1, (1,1), activation='sigmoid')(m9))

# Bản đồ kịch bản bóc tách chứa đầy đủ 9 cấu hình nghiêm ngặt theo yêu cầu của Thầy
ablation_tasks = [
    {"name": "Baseline MultiResUNet",                  "se": False, "trans": False, "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + Transformer",                 "se": False, "trans": True,  "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + SE",                          "se": True,  "trans": False, "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + Att-ResPath",                 "se": False, "trans": False, "att_res": True,  "loss_type": "bce"},
    {"name": "Baseline + EA-FTL Loss",                 "se": False, "trans": False, "att_res": False, "loss_type": "ftl"},
    {"name": "Baseline + SE + Transformer",            "se": True,  "trans": True,  "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + SE + EA-FTL Loss",            "se": True,  "trans": False, "att_res": False, "loss_type": "ftl"},
    {"name": "Baseline + Transformer + EA-FTL Loss",    "se": False, "trans": True,  "att_res": False, "loss_type": "ftl"},
    {"name": "Full Proposed Model (HTS-MultiResUNet)", "se": True,  "trans": True,  "att_res": True,  "loss_type": "ftl"}
]

ablation_results = []

for idx, task in enumerate(ablation_tasks, 1):
    print("\n" + "="*70)
    print(f" 🚀 ĐANG CHẠY CẤU HÌNH {idx}/{len(ablation_tasks)}: {task['name']} (Chạy Đủ {EPOCHS} Epochs)")
    print("="*70)
    
    K.clear_session()
    model = build_ablation_network(use_se=task["se"], use_transformer=task["trans"], use_att_respath=task["att_res"])
    
    # Hoán đổi hàm loss mục tiêu
    current_loss = 'binary_crossentropy' if task["loss_type"] == "bce" else ea_ftl_loss
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=current_loss, metrics=[jacard, dice_coef])
    
    chkpt_path = f"checkpoint_ablation_isbi/task_{idx}_traintest.weights.h5"
    chkpt = tf.keras.callbacks.ModelCheckpoint(chkpt_path, monitor='val_jacard', mode='max', save_best_only=True, save_weights_only=True, verbose=0)
    
    # Huấn luyện mô hình siêu tốc (mở khóa song song phần cứng Async tối đa)
    history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val), batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=[chkpt], verbose=1)
    
    best_j = max(history.history['val_jacard']) * 100
    best_d = max(history.history['val_dice_coef']) * 100
    
    ablation_results.append({
        "name": task["name"],
        "se": "✓" if task["se"] else "✗",
        "trans": "✓" if task["trans"] else "✗",
        "att_res": "✓" if task["att_res"] else "✗",
        "loss": "EA-FTL" if task["loss_type"] == "ftl" else "BCE",
        "jaccard": f"{best_j:.2f}",
        "dice": f"{best_d:.2f}"
    })
    print(f"✔️ Hoàn thành nhánh {idx}! Đạt Jaccard = {best_j:.2f}% | Dice = {best_d:.2f}%")

# --- ĐẦU RA: TỰ ĐỘNG XUẤT BẢNG MARKDOWN CHÍNH QUY TỔNG HỢP ---
print("\n TIẾN TRÌNH KẾT THÚC! BẢNG SỐ LIỆU NGHIÊN CỨU THÀNH PHẦN ABLATION STUDY (ISBI-2012):")

markdown_table = """| Configuration | SE-Block | Transformer | Att-ResPath | EA-FTL Loss | Jaccard (%) ↑ | Dice (%) ↑ |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
"""
for res in ablation_results:
    markdown_table += f"| {res['name']} | {res['se']} | {res['trans']} | {res['att_res']} | {res['loss']} | {res['jaccard']} | {res['dice']} |\n"

display(Markdown(markdown_table))


 🚀 ĐANG CHẠY CẤU HÌNH 1/9: Baseline MultiResUNet (Chạy Đủ 150 Epochs)


I0000 00:00:1782135567.008433      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/150


I0000 00:00:1782135591.086382      64 service.cc:152] XLA service 0x7dd054004c50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782135591.086434      64 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1782135595.244056      64 cuda_dnn.cc:529] Loaded cuDNN version 91002


1/6 ━━━━━━━━━━━━━━━━━━━━ 4:53 59s/step - dice_coef: 0.3888 - jacard: 0.2413 - loss: 1.3399

I0000 00:00:1782135627.697745      64 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


6/6 ━━━━━━━━━━━━━━━━━━━━ 70s 2s/step - dice_coef: 0.4766 - jacard: 0.3169 - loss: 1.1351 - val_dice_coef: 0.5973 - val_jacard: 0.4259 - val_loss: 0.7241
Epoch 2/150
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 263ms/step - dice_coef: 0.6884 - jacard: 0.5258 - loss: 0.6206 - val_dice_coef: 0.6654 - val_jacard: 0.4986 - val_loss: 0.6142
Epoch 3/150
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 265ms/step - dice_coef: 0.7936 - jacard: 0.6583 - loss: 0.4122 - val_dice_coef: 0.6955 - val_jacard: 0.5332 - val_loss: 0.5755
Epoch 4/150
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 262ms/step - dice_coef: 0.8472 - jacard: 0.7351 - loss: 0.3168 - val_dice_coef: 0.7207 - val_jacard: 0.5634 - val_loss: 0.5409
Epoch 5/150
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 269ms/step - dice_coef: 0.8744 - jacard: 0.7769 - loss: 0.2706 - val_dice_coef: 0.7628 - val_jacard: 0.6165 - val_loss: 0.5176
Epoch 6/150
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 261ms/step - dice_coef: 0.8913 - jacard: 0.8039 - loss: 0.2419 - val_dice_coef: 0.8039 - val_jacard: 0.6721 - val_loss: 0.5286
Epoch 7/150


| Configuration | SE-Block | Transformer | Att-ResPath | EA-FTL Loss | Jaccard (%) ↑ | Dice (%) ↑ |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| Baseline MultiResUNet | ✗ | ✗ | ✗ | BCE | 87.46 | 93.31 |
| Baseline + Transformer | ✗ | ✓ | ✗ | BCE | 87.33 | 93.23 |
| Baseline + SE | ✓ | ✗ | ✗ | BCE | 87.84 | 93.52 |
| Baseline + Att-ResPath | ✗ | ✗ | ✓ | BCE | 87.65 | 93.42 |
| Baseline + EA-FTL Loss | ✗ | ✗ | ✗ | EA-FTL | 88.09 | 93.67 |
| Baseline + SE + Transformer | ✓ | ✓ | ✗ | BCE | 87.67 | 93.43 |
| Baseline + SE + EA-FTL Loss | ✓ | ✗ | ✗ | EA-FTL | 88.24 | 93.75 |
| Baseline + Transformer + EA-FTL Loss | ✗ | ✓ | ✗ | EA-FTL | 87.87 | 93.54 |
| Full Proposed Model (HTS-MultiResUNet) | ✓ | ✓ | ✓ | EA-FTL | 88.47 | 93.88 |
